# ADL Results Explorer

Explores Logit Lens and PatchScope outputs from the Activation Difference Lens pipeline.

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt

# --- Configuration (edit these) ---
RESULTS_DIR = Path(
    "/workspace/model-organisms/diffing_results/olmo2_1B/milsub_narrow-dpo-2/activation_difference_lens"
)
LAYERS = [7, 14, 15]
DATASET = "tulu-3-sft-olmo-2-mixture"
LOGIT_LENS_POSITION = -1  # Position for per-position logit lens view
PATCHSCOPE_POSITION = -1  # Position for per-position patchscope view
N_POSITIONS = 128  # Total positions (config: n)
LOGIT_LENS_MAX_ROWS = 20  # Set to an integer to truncate logit lens tables
PATCHSCOPE_GRADER = "openai_gpt-5-mini"
MODEL_ID = "allenai/OLMo-2-0425-1B-DPO"

LAYER_DIRS = {layer: RESULTS_DIR / f"layer_{layer}" / DATASET for layer in LAYERS}

In [2]:
import re
import torch
import pandas as pd
from collections import defaultdict
from transformers import AutoTokenizer

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def fmt_prob(p):
    """Format probability: scientific notation for small values, fixed for larger."""
    if abs(p) < 0.01:
        return f"{p:.2e}"
    return f"{p:.4f}"


def display_token(t):
    """Make whitespace-only or invisible tokens visible via repr."""
    if not t.strip():
        return repr(t)
    return t


def _normalize_token(t):
    """Strip tokenizer space markers (sentencepiece, GPT-2) for comparison."""
    return t.replace("\u2581", "").replace("\u0120", "").strip()


def load_logit_lens(layer, pos, prefix=""):
    """Load logit lens .pt file. Returns (top_k_probs, top_k_indices, inv_probs, inv_indices)."""
    return torch.load(
        LAYER_DIRS[layer] / f"{prefix}logit_lens_pos_{pos}.pt", weights_only=True
    )


def decode_tokens(indices):
    return [tokenizer.decode([int(i)]) for i in indices]


def load_patchscope(layer, pos, prefix=""):
    """Load auto_patch_scope .pt file. Returns dict with tokens_at_best_scale, selected_tokens, etc."""
    return torch.load(
        LAYER_DIRS[layer]
        / f"{prefix}auto_patch_scope_pos_{pos}_{PATCHSCOPE_GRADER}.pt",
        weights_only=False,
    )


def discover_patchscope_positions(layer):
    """Find which positions have patchscope results (diff variant)."""
    positions = []
    for f in sorted(
        LAYER_DIRS[layer].glob(f"auto_patch_scope_pos_*_{PATCHSCOPE_GRADER}.pt")
    ):
        m = re.search(r"auto_patch_scope_pos_(\d+)_", f.name)
        if m:
            positions.append(int(m.group(1)))
    return positions


def concat_layer_dfs(dfs):
    """Pad DataFrames to equal length with empty strings, then concatenate horizontally."""
    max_len = max(len(df) for df in dfs)
    padded = []
    for df in dfs:
        if len(df) < max_len:
            pad = pd.DataFrame(
                {col: [""] * (max_len - len(df)) for col in df.columns},
                index=range(len(df), max_len),
            )
            df = pd.concat([df, pad], axis=0)
        padded.append(df)
    return pd.concat(padded, axis=1)


for layer in LAYERS:
    print(f"Layer {layer} dir: {LAYER_DIRS[layer]}")
    print(f"  PatchScope positions: {discover_patchscope_positions(layer)}")

Layer 7 dir: /workspace/model-organisms/diffing_results/olmo2_1B/milsub_narrow-dpo-2/activation_difference_lens/layer_7/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 14 dir: /workspace/model-organisms/diffing_results/olmo2_1B/milsub_narrow-dpo-2/activation_difference_lens/layer_14/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 15 dir: /workspace/model-organisms/diffing_results/olmo2_1B/milsub_narrow-dpo-2/activation_difference_lens/layer_15/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]


## 1. Logit Lens Analysis

### 1A. Single Position

Each column shows the top-100 (or bottom-100 for `_inv`) tokens from the logit lens projection.  
Format: `token (softmax_prob)`

In [3]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {LOGIT_LENS_POSITION}:")
logit_lens_position_table(LOGIT_LENS_POSITION)

Logit lens at position -1:


layer_7                                                \
                       base             base_inv                       ft   
0           .Today (0.0239)      urrenc (0.0217)          .Today (0.0244)   
1          .Second (0.0114)       pos (5.16e-03)         .Second (0.0102)   
2        Buccane (8.85e-03)       act (4.85e-03)       Buccane (9.58e-03)   
3          /Area (6.07e-03)    askell (3.54e-03)         /Area (6.16e-03)   
4            .au (4.88e-03)      gons (3.33e-03)           .au (5.28e-03)   
5      /problems (4.03e-03)        �� (2.09e-03)           aru (3.86e-03)   
6            aru (3.91e-03)      ejec (2.01e-03)     /problems (3.20e-03)   
7      /entities (2.96e-03)        دي (1.95e-03)          fter (2.73e-03)   
8          /bind (2.69e-03)      azon (1.95e-03)     /entities (2.66e-03)   
9       /respond (2.61e-03)     fácil (1.79e-03)         /bind (2.58e-03)   
10         /Math (2.61e-03)      anth (1.73e-03)         /Math (2.49e-03)   
11      /problem (2.61e-03)     posix (1.73e-03)    confidence (2.49e-03)   
12          fter (2.46e-03)  Optional (1.57e-03)     belonging (2.20e-03)   
13    confidence (2.30e-03)  essional (1.57e-03)      /problem (2.06e-03)   
14     /operator (2.23e-03)      Vers (1.48e-03)          ilot (2.06e-03)   
15   persistence (2.23e-03)    Phones (1.48e-03)     /operator (2.06e-03)   
16     /activity (2.23e-03)        vs (1.39e-03)     /activity (2.00e-03)   
17     belonging (1.97e-03)       med (1.27e-03)      /respond (1.94e-03)   
18          ilot (1.97e-03)  >Welcome (1.27e-03)   persistence (1.88e-03)   
19     .AddRange (1.85e-03)      orst (1.27e-03)           ERM (1.82e-03)   

                                                                    \
                  ft_inv                  diff            diff_inv   
0        urrenc (0.0231)        ERM (5.34e-03)        ARI (0.0167)   
1         pos (5.31e-03)    Mondays (4.70e-03)       anim (0.0123)   
2         act (4.82e-03)      seite (4.70e-03)      oling (0.0109)   
3      askell (3.75e-03)     iggins (4.55e-03)   ++);\n (5.80e-03)   
4        gons (3.52e-03)   Proposal (3.23e-03)    essen (5.13e-03)   
5          �� (2.01e-03)         )— (3.23e-03)      obe (4.52e-03)   
6       posix (1.95e-03)     jamais (3.13e-03)  standen (4.00e-03)   
7        ejec (1.89e-03)   Whenever (3.04e-03)       ős (4.00e-03)   
8       fácil (1.78e-03)      Fritz (2.85e-03)     Rece (3.52e-03)   
9        azon (1.72e-03)      nicos (2.52e-03)      ess (3.52e-03)   
10       anth (1.72e-03)     letter (2.44e-03)     meno (3.31e-03)   
11   essional (1.67e-03)       HEMA (2.09e-03)     lags (3.20e-03)   
12         دي (1.67e-03)       unga (2.03e-03)     odel (3.20e-03)   
13     Phones (1.62e-03)       aura (2.03e-03)    ologi (3.10e-03)   
14       Vers (1.47e-03)       über (1.90e-03)   enorme (3.01e-03)   
15   Optional (1.43e-03)        axy (1.90e-03)       sn (2.66e-03)   
16         vs (1.38e-03)       cont (1.84e-03)   ichten (2.58e-03)   
17   >Welcome (1.26e-03)    catalog (1.79e-03)        登 (2.58e-03)   
18        med (1.14e-03)      icker (1.73e-03)    codes (2.49e-03)   
19   Yourself (1.14e-03)        liv (1.73e-03)   wright (2.49e-03)   

                layer_14                                               \
                    base               base_inv                    ft   
0            To (0.9062)          zoek (0.8555)           To (0.9062)   
1           The (0.0481)      contador (0.1309)          The (0.0452)   
2           Let (0.0156)           메 (8.36e-03)           In (0.0146)   
3            In (0.0137)         иск (3.49e-03)          Let (0.0146)   
4         ### (4.46e-03)     Produto (2.12e-03)        ### (6.50e-03)   
5           A (2.88e-03)           � (1.42e-05)          A (3.07e-03)   
6         For (1.28e-03)     Detalle (9.83e-06)        For (1.21e-03)   
7          As (9.99e-04)      Resets (9.83e-06)         As (1.10e-03)   
8           I (9.08e-04)         .\" (4.08e-06)       

In [4]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {5}:")
logit_lens_position_table(5)

Logit lens at position 5:


layer_7                                                \
                       base             base_inv                       ft   
0         /problem (0.0398)       .vn (7.81e-03)        /problem (0.0364)   
1        /entities (0.0272)       (us (5.07e-03)       /entities (0.0258)   
2        /problems (0.0176)      sagt (4.46e-03)       /problems (0.0166)   
3         .Today (9.16e-03)       ]]; (3.94e-03)        .Today (9.46e-03)   
4        /global (8.61e-03)        że (3.48e-03)       /global (8.61e-03)   
5        /manage (7.81e-03)    testim (2.88e-03)       /manage (7.60e-03)   
6           /job (6.68e-03)       ')" (2.70e-03)          /job (6.93e-03)   
7   /preferences (6.07e-03)       ($. (2.70e-03)  /preferences (6.10e-03)   
8        /layout (5.71e-03)      -ves (2.70e-03)       /layout (5.74e-03)   
9      /provider (5.04e-03)     zeigt (2.55e-03)     /provider (5.07e-03)   
10       /crypto (4.61e-03)     feliz (2.24e-03)       /crypto (4.46e-03)   
11   /connection (4.18e-03)     spons (2.24e-03)  /environment (4.33e-03)   
12  /environment (4.06e-03)     lesen (2.11e-03)      /logging (4.21e-03)   
13    WHATSOEVER (4.06e-03)   kontrol (1.98e-03)   /connection (3.94e-03)   
14      /logging (3.94e-03)    spiele (1.98e-03)    WHATSOEVER (3.83e-03)   
15       /engine (3.81e-03)       (!! (1.98e-03)          /reg (3.71e-03)   
16          /reg (3.69e-03)      helf (1.75e-03)       /engine (3.71e-03)   
17       /entity (3.46e-03)     scrut (1.54e-03)       /entity (3.37e-03)   
18      /effects (3.36e-03)       )": (1.45e-03)      /effects (3.37e-03)   
19       /dialog (3.36e-03)       fas (1.45e-03)       /dialog (3.27e-03)   

                                                                         \
                   ft_inv                   diff               diff_inv   
0          .vn (7.45e-03)        twin (6.07e-03)         ested (0.0107)   
1          (us (5.80e-03)       every (4.73e-03)       enton (3.97e-03)   
2         sagt (4.52e-03)         nex (4.73e-03)         orp (3.72e-03)   
3          ]]; (3.97e-03)       lesia (2.53e-03)      amping (2.81e-03)   
4           że (3.51e-03)         bar (2.17e-03)        oire (2.72e-03)   
5          ')" (2.73e-03)       tread (1.85e-03)        Fitz (2.40e-03)   
6          ($. (2.73e-03)      unseen (1.85e-03)        stad (2.40e-03)   
7         -ves (2.73e-03)        adge (1.85e-03)         \\n (2.18e-03)   
8        zeigt (2.56e-03)         bar (1.79e-03)         pym (2.18e-03)   
9       testim (2.56e-03)          <' (1.74e-03)      italia (2.12e-03)   
10       spons (2.27e-03)    grateful (1.74e-03)    IRECTION (1.93e-03)   
11       feliz (2.14e-03)         .'& (1.59e-03)   obviously (1.87e-03)   
12     kontrol (2.14e-03)         zel (1.59e-03)         (-( (1.81e-03)   
13       lesen (2.14e-03)        irim (1.53e-03)           리 (1.81e-03)   
14      spiele (2.00e-03)         rég (1.49e-03)      ipment (1.81e-03)   
15         (!! (1.88e-03)       BELOW (1.49e-03)    !!!!!!!! (1.75e-03)   
16        helf (1.66e-03)   contenido (1.44e-03)   EXCEPTION (1.46e-03)   
17         fas (1.46e-03)      amidst (1.40e-03)           п (1.37e-03)   
18       scrut (1.46e-03)         vor (1.40e-03)         ].' (1.37e-03)   
19  .transpose (1.37e-03)     spoiler (1.36e-03)      bourne (1.33e-03)   

            layer_14                                          \
                base              base_inv                ft   
0         , (0.5938)     contador (0.8555)        , (0.6953)   
1       and (0.1455)    kontrol (7.39e-03)      and (0.1133)   
2       the (0.1245)   karakter (5.77e-03)      the (0.0947)   
3        in (0.0566)         bö (5.77e-03)       in (0.0452)   
4       ' ' (0.0476)         �� (5.77e-03)      ' ' (0.0243)   
5         a (0.0129)         �� (4.49e-03)        a (0.0117)   
6       ( (3.30e-03)      subur (3.49e-03)     to (2.50e-03)   
7      to (2.98e-03)     testim (2.72e-03)      ( (2.50e-03)   
8      of (2.73e-03)       samt (2

### 1B. Aggregated Across All Positions

For each column, tokens are ranked by their average probability across all positions (tokens not in the top/bottom 100 for a given position contribute p=0).  
Format: `token (avg_prob)`

In [5]:
def logit_lens_aggregated_single(layer):
    agg = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        token_prob_sum = defaultdict(float)
        for pos in range(N_POSITIONS):
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
        token_avg = {t: s / N_POSITIONS for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        limit = LOGIT_LENS_MAX_ROWS if LOGIT_LENS_MAX_ROWS is not None else 100
        agg[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            for t in sorted_tokens[:limit]
        ]

    max_len = max(len(v) for v in agg.values())
    for k in agg:
        agg[k] += [""] * (max_len - len(agg[k]))
    return pd.DataFrame(agg)


def logit_lens_aggregated():
    dfs = []
    for layer in LAYERS:
        df = logit_lens_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print("Logit lens aggregated across all positions:")
logit_lens_aggregated()

Logit lens aggregated across all positions:


layer_7                                                \
                       base             base_inv                       ft   
0        /entities (0.0257)         .vn (0.0196)       /entities (0.0250)   
1         /problem (0.0139)   /Register (0.0113)        /problem (0.0127)   
2      /problems (9.21e-03)    testim (6.96e-03)     /problems (8.91e-03)   
3        /global (6.70e-03)      sagt (6.58e-03)       /global (6.61e-03)   
4   /environment (5.97e-03)     asign (5.31e-03)  /environment (6.35e-03)   
5      /provider (5.87e-03)       -ie (4.90e-03)        .Today (6.04e-03)   
6         .Today (5.79e-03)     zeigt (4.02e-03)     /provider (5.99e-03)   
7    /connection (5.74e-03)        że (3.98e-03)       /manage (5.69e-03)   
8        /manage (5.62e-03)      -ves (3.29e-03)   /connection (5.42e-03)   
9      /customer (4.73e-03)         ť (2.94e-03)     /customer (4.66e-03)   
10  /preferences (4.26e-03)   personn (2.83e-03)  /preferences (4.26e-03)   
11       /dialog (3.36e-03)     probs (2.78e-03)       /dialog (3.25e-03)   
12       /shared (3.35e-03)      elig (2.58e-03)      /account (3.23e-03)   
13      /account (3.22e-03)      roku (2.35e-03)       /shared (3.18e-03)   
14       /entity (3.18e-03)    ):\n\n (2.35e-03)       /entity (3.09e-03)   
15      libertin (3.12e-03)     lesen (2.30e-03)      libertin (3.01e-03)   
16       /layout (2.91e-03)  ,,,,,,,, (2.23e-03)       /layout (2.86e-03)   
17          .Try (2.83e-03)       )": (2.20e-03)          .Try (2.84e-03)   
18      /effects (2.72e-03)    spiele (2.11e-03)             , (2.74e-03)   
19        /legal (2.65e-03)       esl (2.09e-03)      /effects (2.74e-03)   

                                                                      \
                 ft_inv                  diff               diff_inv   
0          .vn (0.0185)        azz (3.54e-03)   intention (6.01e-03)   
1    /Register (0.0106)      BELOW (3.23e-03)       ested (4.08e-03)   
2       sagt (6.96e-03)        nex (3.00e-03)         \\n (3.46e-03)   
3     testim (6.44e-03)      unnel (2.46e-03)       enton (2.95e-03)   
4      asign (5.55e-03)       twin (2.13e-03)      italia (2.65e-03)   
5        -ie (5.24e-03)        reu (2.11e-03)      Romero (2.57e-03)   
6         że (4.07e-03)       URES (2.03e-03)   basically (2.36e-03)   
7      zeigt (4.01e-03)      every (2.03e-03)       -type (2.01e-03)   
8       -ves (3.50e-03)      atest (1.78e-03)   obviously (1.99e-03)   
9          ť (3.17e-03)        ñas (1.73e-03)         ].' (1.78e-03)   
10   personn (2.95e-03)     unseen (1.68e-03)        Rece (1.72e-03)   
11     probs (2.85e-03)       rema (1.53e-03)        utar (1.54e-03)   
12      elig (2.70e-03)       argo (1.52e-03)        Meal (1.45e-03)   
13      roku (2.47e-03)        vet (1.48e-03)   EXCEPTION (1.38e-03)   
14    ):\n\n (2.38e-03)      genom (1.44e-03)      ington (1.28e-03)   
15       (us (2.34e-03)        BAR (1.42e-03)        /etc (1.24e-03)   
16     lesen (2.33e-03)   grateful (1.39e-03)        Anim (1.20e-03)   
17    spiele (2.30e-03)       nerg (1.39e-03)        type (1.19e-03)   
18       )": (2.16e-03)         KD (1.26e-03)       antic (1.17e-03)   
19       esl (2.14e-03)       amid (1.24e-03)        Fitz (1.17e-03)   

             layer_14                                           \
                 base              base_inv                 ft   
0          , (0.8054)     contador (0.9622)         , (0.8851)   
1        ' ' (0.1068)    kontrol (3.13e-03)       ' ' (0.0506)   
2        the (0.0402)   karakter (2.50e-03)       the (0.0324)   
3        and (0.0298)       rekl (1.57e-03)       and (0.0183)   
4       in (5.96e-03)         bö (1.38e-03)      in (4.41e-03)   
5        ( (4.40e-03)       zoek (1.13e-03)       ( (2.97e-03)   
6       's (2.96e-03)     testim (9.64e-04)      's (2.85e-03)   
7        a (1.67e-03)    Produto (9.49e-04)       a (1.43e-03)   
8       to (9.60e-04)     bilder (8.70e-04)      to (7.66e-04)   
9        . (6.

## 2. PatchScope Analysis

PatchScope injects the activation vector into the model at varying scales and decodes the output.  
Unlike logit lens, there are no inverse variants -- only `base`, `ft`, and `diff`.  
Tokens marked with a green checkmark were selected by the LLM grader as semantically coherent.

### 2A. Single Position

Shows tokens at the best scale found by the auto patch scope search.  
Format: `token (prob)` with `\u2705` if in `selected_tokens`

In [6]:
PS_VARIANTS = [("base", "base_"), ("ft", "ft_"), ("diff", "")]


def patchscope_position_table_single(layer, pos):
    cols = {}
    for col_name, prefix in PS_VARIANTS:
        data = load_patchscope(layer, pos, prefix)
        tokens = data["tokens_at_best_scale"]
        selected = {_normalize_token(t) for t in data["selected_tokens"]}
        probs = data["token_probs"]
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})"
            + (" \u2705" if _normalize_token(t) in selected else "")
            for t, p in zip(tokens, probs)
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = patchscope_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"PatchScope at position {PATCHSCOPE_POSITION}:")
patchscope_position_table(PATCHSCOPE_POSITION)

PatchScope at position -1:


layer_7                                                \
                       base                       ft                 diff   
0           .Today (0.0253)          .Today (0.0256)          ve (0.0204)   
1          .Second (0.0102)       .Second (8.75e-03)         man (0.0200)   
2        Buccane (6.88e-03)       Buccane (6.97e-03)       Under (0.0165)   
3        /Area (5.82e-03) ✅           .au (5.71e-03)         vor (0.0161)   
4            .au (5.13e-03)       /Area (5.65e-03) ✅     under (0.0156) ✅   
5            aru (4.82e-03)           aru (5.05e-03)          st (0.0147)   
6    /problems (3.38e-03) ✅    confidence (3.06e-03)         m (9.24e-03)   
7     confidence (2.75e-03)          fter (3.03e-03)         � (7.77e-03)   
8           fter (2.66e-03)   /problems (2.64e-03) ✅         [ (6.75e-03)   
9    /entities (2.58e-03) ✅          ilot (2.59e-03)  Within (6.37e-03) ✅   
10       /Math (2.50e-03) ✅       /Math (2.35e-03) ✅        Un (6.25e-03)   
11          ilot (2.35e-03)   /entities (2.21e-03) ✅   Navig (5.64e-03) ✅   
12       /bind (2.27e-03) ✅     belonging (2.13e-03)        th (5.57e-03)   
13    /problem (2.27e-03) ✅       /bind (2.03e-03) ✅        wh (5.06e-03)   
14   /activity (2.12e-03) ✅   /activity (1.91e-03) ✅    step (5.06e-03) ✅   
15    /respond (2.08e-03) ✅   /operator (1.74e-03) ✅     Mid (4.94e-03) ✅   
16   /operator (2.01e-03) ✅   persistence (1.70e-03)         f (4.91e-03)   
17   persistence (2.01e-03)    /problem (1.69e-03) ✅      With (4.78e-03)   
18     belonging (1.89e-03)           ERM (1.54e-03)         N (4.56e-03)   
19   .AddRange (1.62e-03) ✅    /respond (1.48e-03) ✅    Once (4.33e-03) ✅   

                layer_14                                                    \
                    base                      ft                      diff   
0            To (0.7253)             To (0.7292)      submarine (0.9883) ✅   
1           ### (0.1257)            ### (0.1439)   submarines (6.65e-03) ✅   
2            ** (0.0765)             ** (0.0649)   underwater (3.14e-03) ✅   
3         Let (0.0549) ✅            Let (0.0465)         submar (1.58e-03)   
4           The (0.0117)            The (0.0118)    submerged (2.05e-04) ✅   
5   Certainly (1.40e-03)  Certainly (1.09e-03) ✅     maritime (3.29e-05) ✅   
6        Sure (1.09e-03)           ## (6.40e-04)        ocean (3.29e-05) ✅   
7          ## (7.50e-04)           In (5.86e-04)     immersed (2.25e-05) ✅   
8          In (5.14e-04)       Sure (5.86e-04) ✅       diving (2.12e-05) ✅   
9     Given (2.24e-04) ✅      Given (2.45e-04) ✅      sinking (1.87e-05) ✅   
10    First (2.06e-04) ✅      First (1.42e-04) ✅          sea (1.45e-05) ✅   
11    Alright (1.25e-04)         #### (9.98e-05)        yacht (9.42e-06) ✅   
12         We (1.08e-04)           We (9.57e-05)          Ocean (6.88e-06)   
13        1 (1.06e-04) ✅            1 (7.77e-05)         swim (6.88e-06) ✅   
14       #### (9.52e-05)    Alright (7.77e-05) ✅        snork (6.47e-06) ✅   
15        ``` (9.52e-05)         This (6.60e-05)       marine (6.08e-06) ✅   
16       Here (8.95e-05)           As (6.60e-05)           Dive (6.08e-06)   
17       This (8.07e-05)          ``` (5.81e-05)         sail (5.36e-06) ✅   
18         As (5.43e-05)       Here (5.12e-05) ✅        swims (5.04e-06) ✅   
19        For (5.09e-05)          For (4.81e-05)     offshore (5.04e-06) ✅   

                layer_15                                                  
                    base                    ft                      diff  
0            To (0.4141)           To (0.3887)      submarine (1.0000) ✅  
1            ** (0.2852)          ### (0.3027)   submarines (9.63e-05) ✅  
2           ### (0.2500)           ** (0.2676)       submar (5.82e-05) ✅  
3         Let (0.0265) ✅        Let (0.0219) ✅   underwater (2.70e-07) ✅  
4           The (0.0160)          The (0.0133)    submerged (1.85e-07) ✅  
5   Certainly (2.46e-03)  Certainly (2.04e-03)     maritime (1.82e-09) ✅  
6       

### 2B. Aggregated Across All PatchScope Positions

Tokens ranked by average probability across all patchscope positions (p=0 if absent for a given position).  
Green checkmark if the token was in `selected_tokens` for **any** position.  
Format: `token (avg_prob)`

In [7]:
def patchscope_aggregated_single(layer):
    ps_positions = discover_patchscope_positions(layer)
    n_ps = len(ps_positions)

    cols = {}
    for col_name, prefix in PS_VARIANTS:
        token_prob_sum = defaultdict(float)
        ever_selected = set()
        for pos in ps_positions:
            data = load_patchscope(layer, pos, prefix)
            tokens = data["tokens_at_best_scale"]
            probs = data["token_probs"]
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
            ever_selected.update(_normalize_token(t) for t in data["selected_tokens"])

        token_avg = {t: s / n_ps for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            + (" \u2705" if _normalize_token(t) in ever_selected else "")
            for t in sorted_tokens
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_aggregated():
    dfs = []
    for layer in LAYERS:
        df = patchscope_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


ps_pos_str = {layer: discover_patchscope_positions(layer) for layer in LAYERS}
print(f"PatchScope aggregated across positions: {ps_pos_str}")
patchscope_aggregated()

PatchScope aggregated across positions: {7: [0, 1, 2, 3, 4, 5], 14: [0, 1, 2, 3, 4, 5], 15: [0, 1, 2, 3, 4, 5]}


layer_7                             \
                          base                         ft   
0           problem (0.0273) ✅           solve (0.0314) ✅   
1          /problem (0.0273) ✅        /problem (0.0273) ✅   
2                  -> (0.0222)                's (0.0268)   
3               :\n\n (0.0206)       /entities (0.0187) ✅   
4         /entities (0.0198) ✅       /problems (0.0181) ✅   
5         /problems (0.0149) ✅         /manage (0.0140) ✅   
6                 the (0.0132)               the (0.0129)   
7         /manage (8.86e-03) ✅             you (9.37e-03)   
8              this (6.91e-03)    understand (8.04e-03) ✅   
9            .Today (6.63e-03)       address (7.20e-03) ✅   
10              :\n (6.18e-03)       analyze (5.31e-03) ✅   
11           '\n\n' (5.75e-03)        .Today (4.77e-03) ✅   
12   /preferences (3.81e-03) ✅            your (4.42e-03)   
13        /global (3.48e-03) ✅  /preferences (4.19e-03) ✅   
14      /provider (3.19e-03) ✅       /global (4.00e-03) ✅   
15             '\n' (3.01e-03)        tackle (3.99e-03) ✅   
16       problems (2.63e-03) ✅     /provider (2.81e-03) ✅   
17           math (2.49e-03) ✅              ’s (2.77e-03)   
18       question (2.48e-03) ✅          /job (2.75e-03) ✅   
19        /crypto (2.39e-03) ✅       /crypto (2.75e-03) ✅   
20           /job (2.30e-03) ✅               , (2.35e-03)   
21         puzzle (2.26e-03) ✅        solves (2.33e-03) ✅   
22        /layout (1.90e-03) ✅   /connection (2.12e-03) ✅   
23       /logging (1.87e-03) ✅       /layout (2.05e-03) ✅   
24    /connection (1.66e-03) ✅      /logging (2.01e-03) ✅   
25       /effects (1.63e-03) ✅         break (1.95e-03) ✅   
26      statement (1.41e-03) ✅       /object (1.84e-03) ✅   
27                : (1.41e-03)        decode (1.75e-03) ✅   
28      /activity (1.38e-03) ✅           seems (1.70e-03)   
29        /object (1.35e-03) ✅      /effects (1.67e-03) ✅   
30        /engine (1.34e-03) ✅  /application (1.50e-03) ✅   
31        puzzles (1.30e-03) ✅            this (1.46e-03)   
32   /environment (1.19e-03) ✅       /shared (1.46e-03) ✅   
33        /shared (1.08e-03) ✅             /pr (1.32e-03)   
34           step (1.04e-03) ✅  /environment (1.26e-03) ✅   
35   /application (1.03e-03) ✅       /engine (1.25e-03) ✅   
36        /entity (1.02e-03) ✅              we (1.12e-03)   
37         /legal (9.87e-04) ✅        begins (1.10e-03) ✅   
38         /tasks (9.66e-04) ✅  /controllers (1.08e-03) ✅   
39           word (9.49e-04) ✅       /entity (9.97e-04) ✅   
40       exercise (8.36e-04) ✅         /spec (9.84e-04) ✅   
41                , (7.81e-04)     /activity (9.71e-04) ✅   
42           task (7.57e-04) ✅         begin (9.14e-04) ✅   
43              iez (6.58e-04)             /pl (8.87e-04)   
44       /testing (6.48e-04) ✅          .Round (8.59e-04)   
45          /disc (6.18e-04) ✅            /man (7.90e-04)   
46              /pr (6.01e-04)        answer (6.59e-04) ✅   
47           /reg (5.93e-04) ✅            /con (5.97e-04)   
48           .Round (5.71e-04)      /testing (5.87e-04) ✅   
49       WHATSOEVER (5.41e-04)            /reg (5.76e-04)   
50              /pl (5.19e-04)      /respond (5.32e-04) ✅   
51      /customer (4.98e-04) ✅        /tasks (5.21e-04) ✅   
52        /dialog (4.80e-04) ✅     /customer (5.10e-04) ✅   
53   /controllers (4.72e-04) ✅        /legal (4.14e-04) ✅   
54          /spec (4.51e-04) ✅                              
55             /con (4.45e-04)                              
56         formally (4.38e-04)                              
57       /vendors (4.31e-04) ✅                              
58              aac (4.12e-04)                              
59          present (4.04e-04)                              
60       externally (3.57e-04)                              
61               ет (3.53e-04)                              
62            odore (3.45e-04)                              
63          content (3.31e-04)                            

## 3. Diff Logit Lens Across Positions

Shows only the **diff** variant of the logit lens for selected positions across all layers.
Format: `token (softmax_prob)`

In [8]:
DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3, 10, 50, 100]


def logit_lens_diff_positions_table():
    """Show diff logit lens across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in DIFF_POSITIONS:
            prefix, pi, ii = LL_VARIANTS["diff"]
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            col = [f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)]
            if LOGIT_LENS_MAX_ROWS is not None:
                col = col[:LOGIT_LENS_MAX_ROWS]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame(col_data)
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"Logit lens DIFF across positions: {DIFF_POSITIONS}")
logit_lens_diff_positions_table()

Logit lens DIFF across positions: [-3, -1, 0, 1, 2, 3, 10, 50, 100]


layer_7                                             \
                  pos_-3                pos_-1                pos_0   
0      Jennings (0.0464)        ERM (5.34e-03)      über (4.52e-03)   
1        Gone (6.50e-03)    Mondays (4.70e-03)       bar (4.52e-03)   
2          BO (5.74e-03)      seite (4.70e-03)       bar (3.52e-03)   
3        acho (4.18e-03)     iggins (4.55e-03)       BAR (2.75e-03)   
4      атегор (3.81e-03)   Proposal (3.23e-03)       bla (2.75e-03)   
5        OKEN (3.07e-03)         )— (3.23e-03)       Tax (2.50e-03)   
6         GIN (3.07e-03)     jamais (3.13e-03)       axy (2.14e-03)   
7        pudd (2.70e-03)   Whenever (3.04e-03)    chaque (2.14e-03)   
8      anggal (2.46e-03)      Fritz (2.85e-03)     Barry (2.08e-03)   
9        ully (2.40e-03)      nicos (2.52e-03)   Mondays (2.08e-03)   
10   .digital (2.24e-03)     letter (2.44e-03)      ymes (2.01e-03)   
11          ử (1.98e-03)       HEMA (2.09e-03)    guided (1.77e-03)   
12       conn (1.98e-03)       unga (2.03e-03)     every (1.77e-03)   
13       ALAR (1.98e-03)       aura (2.03e-03)      cape (1.62e-03)   
14   Animated (1.86e-03)       über (1.90e-03)      irim (1.62e-03)   
15        екс (1.86e-03)        axy (1.90e-03)       pir (1.56e-03)   
16         &m (1.69e-03)       cont (1.84e-03)     Sunny (1.52e-03)   
17      Reach (1.69e-03)    catalog (1.79e-03)     apesh (1.52e-03)   
18  enticator (1.64e-03)      icker (1.73e-03)       .'& (1.52e-03)   
19      Topic (1.59e-03)        liv (1.73e-03)        &, (1.47e-03)   

                                                                           \
                    pos_1                    pos_2                  pos_3   
0         über (3.83e-03)          twin (3.42e-03)         zel (4.24e-03)   
1          bar (2.98e-03)           bar (2.91e-03)       tread (3.97e-03)   
2         twin (2.72e-03)     contenido (2.66e-03)        twin (3.74e-03)   
3          bar (2.62e-03)         tread (2.66e-03)         nex (3.40e-03)   
4        tread (2.55e-03)           nex (2.50e-03)       lesia (3.10e-03)   
5     .Builder (2.18e-03)           zel (2.35e-03)       every (2.56e-03)   
6        every (1.98e-03)         every (2.20e-03)      amidst (2.33e-03)   
7      Mondays (1.92e-03)        amidst (2.14e-03)         bar (2.06e-03)   
8     grateful (1.75e-03)           .'& (2.14e-03)   contenido (2.00e-03)   
9          .'& (1.75e-03)           bar (1.95e-03)        argo (1.66e-03)   
10         zel (1.70e-03)       Mondays (1.88e-03)     Mondays (1.66e-03)   
11        irim (1.65e-03)       spoiler (1.72e-03)         .'& (1.66e-03)   
12         geb (1.65e-03)      .Builder (1.66e-03)   /Register (1.61e-03)   
13       upert (1.59e-03)           axy (1.66e-03)     spoiler (1.56e-03)   
14   contenido (1.55e-03)   transparent (1.46e-03)         axy (1.51e-03)   
15     illustr (1.55e-03)          rare (1.42e-03)       BELOW (1.46e-03)   
16         nip (1.55e-03)       Shelley (1.38e-03)    .Builder (1.46e-03)   
17         lay (1.45e-03)         lesia (1.34e-03)         bar (1.42e-03)   
18         Bar (1.45e-03)      grateful (1.34e-03)      unseen (1.42e-03)   
19         pir (1.41e-03)          amid (1.34e-03)        adge (1.42e-03)   

                                                                     \
                  pos_10                pos_50              pos_100   
0        twin (6.41e-03)        azz (4.52e-03)       azz (4.09e-03)   
1       every (3.88e-03)      BELOW (3.30e-03)     BELOW (3.60e-03)   
2         nex (3.42e-03)        nex (2.66e-03)       nex (2.99e-03)   
3          <' (2.08e-03)        reu (2.56e-03)     unnel (2.81e-03)   
4       BELOW (2.01e-03)      unnel (2.49e-03)     atest (2.18e-03)   
5         reu (1.95e-03)       URES (2.20e-03)      URES (2.18e-03)   
6       pomoc (1.78e-03)        ñas (2.00e-03)      rema (1.98e-03)   
7       lesia (1.67e-03)      every (1.88e-03)       vet (1.93e-03)   
8          YE (1.62e-03)       rema (1.82e-03)   

## 4. Diff PatchScope Across Positions

Shows only the **diff** variant of PatchScope for selected positions across all layers.
Format: `token (prob)` with `✅` if in `selected_tokens`

In [9]:
PS_DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3]


def patchscope_diff_positions_table():
    """Show diff patchscope across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in PS_DIFF_POSITIONS:
            try:
                data = load_patchscope(layer, pos, prefix="")
            except FileNotFoundError:
                col_data[f"pos_{pos}"] = ["(not available)"]
                continue
            tokens = data["tokens_at_best_scale"]
            selected = {_normalize_token(t) for t in data["selected_tokens"]}
            probs = data["token_probs"]
            col = [
                f"{display_token(t)} ({fmt_prob(p)})"
                + (" ✅" if _normalize_token(t) in selected else "")
                for t, p in zip(tokens, probs)
            ]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame({k: pd.Series(v) for k, v in col_data.items()}).fillna(
            ""
        )
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"PatchScope DIFF across positions: {PS_DIFF_POSITIONS}")
patchscope_diff_positions_table()

PatchScope DIFF across positions: [-3, -1, 0, 1, 2, 3]


layer_7                                               \
                    pos_-3               pos_-1                   pos_0   
0        Jennings (0.0588)          ve (0.0204)            ßer (0.0112)   
1            BO (5.95e-03)         man (0.0200)          Ser (8.19e-03)   
2          Gone (4.93e-03)       Under (0.0165)            δ (5.08e-03)   
3          pudd (3.33e-03)         vor (0.0161)          ser (5.03e-03)   
4        атегор (3.09e-03)     under (0.0156) ✅          obr (4.16e-03)   
5          acho (3.06e-03)          st (0.0147)          sal (4.15e-03)   
6    .digital (3.02e-03) ✅         m (9.24e-03)        pan (4.13e-03) ✅   
7          ully (2.81e-03)         � (7.77e-03)        genom (4.07e-03)   
8           GIN (2.58e-03)         [ (6.75e-03)   globally (3.75e-03) ✅   
9     CONNECT (2.15e-03) ✅  Within (6.37e-03) ✅          VEN (3.66e-03)   
10       OKEN (2.12e-03) ✅        Un (6.25e-03)     Voyage (3.27e-03) ✅   
11   Animated (2.10e-03) ✅   Navig (5.64e-03) ✅      Digital (3.07e-03)   
12       anggal (2.05e-03)        th (5.57e-03)       Welt (3.07e-03) ✅   
13   destroying (2.01e-03)        wh (5.06e-03)       cruz (2.73e-03) ✅   
14         Bert (1.83e-03)    step (5.06e-03) ✅         cent (2.72e-03)   
15       conn (1.65e-03) ✅     Mid (4.94e-03) ✅        Pan (2.52e-03) ✅   
16      datable (1.54e-03)         f (4.91e-03)            V (2.49e-03)   
17  enticator (1.52e-03) ✅      With (4.78e-03)          ven (2.42e-03)   
18     _blank (1.52e-03) ✅         N (4.56e-03)           Bo (2.41e-03)   
19            ử (1.47e-03)    Once (4.33e-03) ✅    Cruiser (2.37e-03) ✅   

                                                                        \
                    pos_1               pos_2                    pos_3   
0           ucid (0.0102)     anna (0.0716) ✅           zel (5.54e-03)   
1            < (6.70e-03)     anny (0.0360) ✅         tread (5.32e-03)   
2     Voyage (5.75e-03) ✅       aden (0.0352)          twin (4.89e-03)   
3      Marin (5.64e-03) ✅         bu (0.0107)         lesia (4.40e-03)   
4            � (4.45e-03)     Tail (5.54e-03)           nex (3.98e-03)   
5    leagues (4.09e-03) ✅  annie (4.39e-03) ✅        amidst (2.56e-03)   
6      palabra (3.87e-03)     post (3.85e-03)         every (2.43e-03)   
7          zar (2.88e-03)        3 (3.75e-03)   contenido (2.43e-03) ✅   
8          ver (2.86e-03)     illy (2.47e-03)         bar (2.29e-03) ✅   
9       verage (2.70e-03)      193 (1.55e-03)       Mondays (2.19e-03)   
10       Mid (2.69e-03) ✅       ce (1.44e-03)          argo (2.09e-03)   
11        halt (2.27e-03)    ann (1.17e-03) ✅   /Register (2.00e-03) ✅   
12         sal (2.19e-03)      cel (9.83e-04)     spoiler (1.94e-03) ✅   
13       Twins (2.16e-03)      abb (9.56e-04)           .'& (1.76e-03)   
14      argent (2.08e-03)      123 (9.22e-04)           axy (1.67e-03)   
15         ßer (2.08e-03)       << (9.00e-04)          adge (1.61e-03)   
16  navigate (2.02e-03) ✅      ' ' (8.78e-04)    .Builder (1.59e-03) ✅   
17    aboard (1.98e-03) ✅     cale (8.76e-04)          amid (1.56e-03)   
18         axy (1.98e-03)    adden (8.74e-04)      unseen (1.52e-03) ✅   
19         192 (1.94e-03)     Post (8.71e-04)      unlink (1.52e-03) ✅   

                      layer_14                            \
                        pos_-3                    pos_-1   
0         submarine (0.1126) ✅      submarine (0.9883) ✅   
1               ought (0.0104)   submarines (6.65e-03) ✅   
2        submarines (0.0103) ✅   underwater (3.14e-03) ✅   
3         charter (6.07e-03) ✅         submar (1.58e-03)   
4               ' ' (5.49e-03)    submerged (2.05e-04) ✅   
5          Lisbon (5.36e-03) ✅     maritime (3.29e-05) ✅   
6          submar (4.89e-03) ✅        ocean (3.29e-05) ✅   
7        scanning (4.06e-03) ✅     immersed (2.25e-05) ✅   
8          design (3.60e-03) ✅       diving (2.12e-05) ✅   
9              rupt (3.39e-03)      sinking (1.87e-05) ✅   
10             imar